In [0]:
%pip install hana_ml
%pip install openai

In [0]:
import hana_ml.dataframe as dataframe
conn = dataframe.ConnectionContext(address='0683fdd1-9cc5-4454-8ec3-953e104e4985.hna0.prod-us10.hanacloud.ondemand.com',
                                   port=443,
                                   user='BDFB93D9BCA74AB08A28DBFEAD7853B1_E3ARUSX6FKDEATBTRFH9RWP41_RT',
                                   password='By6urVYz-_Z4mws4b4jODRsEoMH4LWtDBTFh0lQt4.j9SsVjG_Gm-TSz-gDle7nkVD048VtIr3C.ee_QxNOz.RNxtIibaXVxOmp7fuXGDMx33N_cJzz2H9vmBOjsVlgr')
conn.connection.isconnected()
 
#Data exploration
# Point to the table that contains the data

df_remote = (conn.table('cost_allocation', schema='BDFB93D9BCA74AB08A28DBFEAD7853B1'))
 
# Verify the column types
 
pdf = df_remote.collect()               # pulls data to driver as pandas DataFrame
df  = spark.createDataFrame(pdf)

display(df)
#new_records = [
#    {'COMPANY': 'PHC1', 'COSTCENTER': 'PH003', 'DESC': 'SALARY_COST', 'AMOUNT': 8000},
#    {'COMPANY': 'PHC2', 'COSTCENTER': 'PH004', 'DESC': 'BONUS_COST', 'AMOUNT': 5000}
#]




In [0]:
from openai import OpenAI
import os

# How to get your Databricks token: https://docs.databricks.com/en/dev-tools/auth/pat.html
#DATABRICKS_TOKEN = os.environ.get('DATABRICKS_TOKEN')
# Alternatively in a Databricks notebook you can use this:
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url="https://enb-deloitte.cloud.databricks.com/serving-endpoints"
)

user_input ='distribute $200 among manufacturing, canteen, salary in 5:2:3'
system_message = 'your task is to allocate cost between cost centers based on instruction given in how to do cost allocation. In Prompt, you will have Existing cost allocated to cost centers and also you will have instruction how to distribute new cost between cost centers. after you process allocation, strictly you will return the updated cost allocation table in this format for all affacted cost centers for example: [ {\'COMPANY\': \'PHC1\', \'COSTCENTER\': \'PH003\', \'DESC\': \'SALARY_COST\', \'AMOUNT\': 8000}, {\'COMPANY\': \'PHC2\', \'COSTCENTER\': \'PH004\', \'DESC\': \'BONUS_COST\', \'AMOUNT\': 5000}]. Strictly follow exact 100% Allocation for distribution amount among cost centers. should not be  more than 100% and should not be less than 100%'

prompt = 'Existing cost is as follows: \n' + df.toPandas().to_string() + '\n' + 'How to do cost allocation: \n' + user_input

response = client.chat.completions.create(
    model="databricks-claude-sonnet-4-5",
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ],
    max_tokens=5000
)

print(response)

In [0]:
import re
import json

# Get content from GPT response
content = response.choices[0].message.content

json_pattern = r'\[\s*\{.*?\}\s*\]'
match = re.search(json_pattern, content, re.DOTALL)

if match:
    json_str = match.group(0)
    # Replace single quotes with double quotes for valid JSON
    json_str = json_str.replace("'", '"')
    # Parse the JSON string
    new_records = json.loads(json_str)
    print("Extracted Records:")
    print(new_records)
else:
    print("No JSON array found in response")

In [0]:
# Merge multiple new records in HANA table
print(new_records)
values_sql = "union all\n".join(
    f"SELECT '{rec['COMPANY']}' AS COMPANY, '{rec['COSTCENTER']}' AS COSTCENTER, '{rec['DESC']}' AS DESC, {rec['AMOUNT']} AS AMOUNT FROM dummy "
    for rec in new_records
)
merge_sql = f"""
MERGE INTO "BDFB93D9BCA74AB08A28DBFEAD7853B1"."cost_allocation" AS target
USING (
    {values_sql}
) AS source
ON (target.COMPANY = source.COMPANY AND target.COSTCENTER = source.COSTCENTER)
WHEN MATCHED THEN
    UPDATE SET target.DESC = source.DESC, target.AMOUNT = source.AMOUNT
WHEN NOT MATCHED THEN
    INSERT (COMPANY, COSTCENTER, DESC, AMOUNT)
    VALUES (source.COMPANY, source.COSTCENTER, source.DESC, source.AMOUNT);
"""
print(merge_sql)
conn.connection.cursor().execute(merge_sql)

pdf1 = df_remote.collect()               # pulls data to driver as pandas DataFrame
df1  = spark.createDataFrame(pdf1)
display(df1)